<a href="https://colab.research.google.com/github/naylaakas/PembelajaranMesin-2026/blob/main/JS06/Praktikum3-HNSW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679069 sha256=f30e665d499c3f09e94db368f79109980d2124cdfcdf6a419b80335a780a6c72
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib


In [2]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

# ===========================
# 1. Buat data 2D acak
# ===========================
num_elements = 1000
dim = 2
data = np.random.random((num_elements, dim)).astype(np.float32)

# Query point
query = np.array([[0.5, 0.5]], dtype=np.float32)
k = 5  # cari 5 tetangga terdekat

# ===========================
# 2. Exact NN (Brute Force)
# ===========================
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

start = time.time()
distances, indices = nn.kneighbors(query)
end = time.time()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Waktu:", end - start, "detik")

# ===========================
# 3. HNSW
# ===========================
# Inisialisasi index HNSW
p = hnswlib.Index(space='l2', dim=dim)

# Ukuran maksimum elemen yang bisa ditampung
p.init_index(max_elements=num_elements, ef_construction=100, M=16)

# Tambahkan data
p.add_items(data)

# Set parameter pencarian
p.set_ef(50)   # tradeoff speed vs accuracy

start = time.time()
labels, distances = p.knn_query(query, k=k)
end = time.time()

print("\n=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances)
print("Waktu:", end - start, "detik")


=== Exact NN ===
Indices: [[819 436  54 103 389]]
Distances: [[0.0135     0.02177473 0.0289792  0.05007425 0.05403187]]
Waktu: 0.03205513954162598 detik

=== HNSW ===
Indices: [[819 436  54 103 389]]
Distances: [[0.00018225 0.00047414 0.00083979 0.00250743 0.00291944]]
Waktu: 0.0001418590545654297 detik


In [3]:

# Konfigurasi percobaan Angular/Cosine
data_sizes = [1000, 1000000]
dimensions = [2, 5]

k = 5
results = []

# Melakukan semua kombinasi percobaan
for num_elements in data_sizes:
    for dim in dimensions:

        print(
            f"\nMetrik: Angular/Cosine | "
            f"Jumlah data: {num_elements} | Dimensi: {dim}D"
        )

        # 1. Membuat dataset
        rng = np.random.default_rng(42 + num_elements + dim)

        data = rng.random(
            (num_elements, dim)
        ).astype(np.float32)

        # Query point
        query = np.full(
            (1, dim), 0.5, dtype=np.float32
        )

        # 2. Exact NN menggunakan Brute Force
        nn = NearestNeighbors(
            n_neighbors=k,
            algorithm="brute",
            metric="cosine"
        )

        nn.fit(data)

        start = time.time()
        distances_exact, indices_exact = nn.kneighbors(query)
        end = time.time()

        time_exact = end - start

        # 3. HNSW menggunakan Cosine
        p = hnswlib.Index(
            space="cosine",
            dim=dim
        )

        p.init_index(
            max_elements=num_elements,
            ef_construction=100,
            M=16
        )

        p.add_items(data)
        p.set_ef(50)

        start = time.time()
        labels, distances_hnsw = p.knn_query(query, k=k)
        end = time.time()

        time_hnsw = end - start

        # 4. Menampilkan hasil percobaan
        print("=== Exact NN ===")
        print("Indices:", indices_exact[0])
        print("Distances:", distances_exact[0])
        print("Waktu:", round(time_exact * 1000, 4), "ms")

        print("\n=== HNSW ===")
        print("Indices:", labels[0])
        print("Distances:", distances_hnsw[0])
        print("Waktu:", round(time_hnsw * 1000, 4), "ms")

        # 5. Menyimpan hasil untuk tabel
        results.append([
            "Angular/Cosine",
            num_elements,
            f"{dim}D",
            indices_exact[0].tolist(),
            labels[0].tolist(),
            round(time_exact * 1000, 4),
            round(time_hnsw * 1000, 4)
        ])

        # Membersihkan variabel percobaan
        del data, query, nn, p

# 6. Menampilkan tabel hasil percobaan
print("\nTABEL HASIL PRAKTIKUM 3 - ANGULAR/COSINE")
print("=" * 125)

print(
    f"{'Metrik':<18}"
    f"{'Jumlah Data':<15}"
    f"{'Dimensi':<10}"
    f"{'Exact NN Index':<30}"
    f"{'HNSW Index':<30}"
    f"{'Waktu Exact (ms)':<17}"
    f"{'Waktu HNSW (ms)':<17}"
)

print("-" * 125)

for row in results:
    print(
        f"{row[0]:<18}"
        f"{row[1]:<15}"
        f"{row[2]:<10}"
        f"{str(row[3]):<30}"
        f"{str(row[4]):<30}"
        f"{row[5]:<17}"
        f"{row[6]:<17}"
    )

print("=" * 125)



Metrik: Angular/Cosine | Jumlah data: 1000 | Dimensi: 2D
=== Exact NN ===
Indices: [161 764 204 834 756]
Distances: [0.0000000e+00 0.0000000e+00 5.9604645e-08 4.1723251e-07 4.7683716e-07]
Waktu: 9.9075 ms

=== HNSW ===
Indices: [161 764 204 834 756]
Distances: [0.0000000e+00 0.0000000e+00 5.9604645e-08 3.5762787e-07 5.9604645e-07]
Waktu: 0.0553 ms

Metrik: Angular/Cosine | Jumlah data: 1000 | Dimensi: 5D
=== Exact NN ===
Indices: [734 784 336 274 803]
Distances: [0.00163341 0.00180531 0.00343889 0.00395012 0.00460112]
Waktu: 1.229 ms

=== HNSW ===
Indices: [734 784 336 274 803]
Distances: [0.00163341 0.00180537 0.00343889 0.00395024 0.004601  ]
Waktu: 0.0618 ms

Metrik: Angular/Cosine | Jumlah data: 1000000 | Dimensi: 2D
=== Exact NN ===
Indices: [ 90464 395038 768525  63228 509327]
Distances: [0. 0. 0. 0. 0.]
Waktu: 22.9487 ms

=== HNSW ===
Indices: [363901 418934 535016 687761 804130]
Distances: [-1.1920929e-07 -1.1920929e-07 -1.1920929e-07  0.0000000e+00
  0.0000000e+00]
Waktu: 0.4